# 01 — Embeddings

Generates and caches TF-IDF, MiniLM, RoBERTa (frozen), and OpenAI embeddings.
TF-IDF/MiniLM/OpenAI run on the (sampled) train set and full test set.
RoBERTa runs on its own smaller `ROBERTA_SAMPLE_SIZE` cap (both train and
test) since CPU inference for a transformer is far slower than the other
methods. Downstream notebooks load from `embeddings_cache/` rather than
recomputing.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import numpy as np
import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.embeddings import (
    get_tfidf_embeddings,
    get_sentence_embeddings,
    get_bert_embeddings,
    get_openai_embeddings,
)

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

train_sample = stratified_sample(train_clean, config.SAMPLE_SIZE, seed=config.SEED)
suffix = f"n{config.SAMPLE_SIZE}" if config.SAMPLE_SIZE else "full"

roberta_train_sample = stratified_sample(train_clean, config.ROBERTA_SAMPLE_SIZE, seed=config.SEED)
roberta_test_sample = stratified_sample(test_clean, config.ROBERTA_SAMPLE_SIZE, seed=config.SEED)
roberta_suffix = f"n{config.ROBERTA_SAMPLE_SIZE}" if config.ROBERTA_SAMPLE_SIZE else "full"

print(f"Embedding {len(train_sample)} train rows ({suffix}) and {len(test_clean)} test rows "
      f"for tfidf/minilm/openai")
print(f"Embedding {len(roberta_train_sample)} train rows and {len(roberta_test_sample)} test rows "
      f"({roberta_suffix}) for roberta")

Embedding 8000 train rows (n8000) and 7600 test rows for tfidf/minilm/openai
Embedding 500 train rows and 500 test rows (n500) for roberta


In [3]:
train_tfidf = get_tfidf_embeddings(train_sample["text"].tolist(), cache_name=f"tfidf_train_{suffix}")
test_tfidf = get_tfidf_embeddings(test_clean["text"].tolist(), cache_name="tfidf_test_full")
assert train_tfidf.shape[0] == len(train_sample)
assert test_tfidf.shape[0] == len(test_clean)
print("TF-IDF dims:", train_tfidf.shape[1])

TF-IDF dims: 5000


In [4]:
train_minilm = get_sentence_embeddings(train_sample["text"].tolist(), cache_name=f"minilm_train_{suffix}")
test_minilm = get_sentence_embeddings(test_clean["text"].tolist(), cache_name="minilm_test_full")
assert train_minilm.shape[0] == len(train_sample)
print("MiniLM dims:", train_minilm.shape[1])

MiniLM dims: 384


In [5]:
train_roberta = get_bert_embeddings(roberta_train_sample["text"].tolist(), cache_name=f"roberta_train_{roberta_suffix}")
test_roberta = get_bert_embeddings(roberta_test_sample["text"].tolist(), cache_name=f"roberta_test_{roberta_suffix}")
assert train_roberta.shape[0] == len(roberta_train_sample)
assert test_roberta.shape[0] == len(roberta_test_sample)
print("RoBERTa dims:", train_roberta.shape[1])

RoBERTa dims: 768


In [6]:
if config.OPENAI_API_KEY:
    try:
        train_openai = get_openai_embeddings(train_sample["text"].tolist(), cache_name=f"openai_train_{suffix}")
        test_openai = get_openai_embeddings(test_clean["text"].tolist(), cache_name="openai_test_full")
        assert train_openai.shape[0] == len(train_sample)
        print("OpenAI dims:", train_openai.shape[1])
    except Exception as e:
        print(f"OpenAI embedding failed (likely quota/billing issue): {type(e).__name__}: {str(e)}")
        print("Skipping OpenAI embeddings.")
else:
    print("OPENAI_API_KEY not set in .env — skipping OpenAI embeddings.")

OpenAI embedding failed (likely quota/billing issue): RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Skipping OpenAI embeddings.


In [7]:
for name, arr in [("tfidf", train_tfidf), ("minilm", train_minilm), ("roberta", train_roberta)]:
    assert not np.isnan(arr).any(), f"{name} embeddings contain NaNs"
print("No NaNs in any computed embedding matrix.")

No NaNs in any computed embedding matrix.
